# 06 Ensemble without LUNAR v2

In [ ]:

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

sys.path.append("../src")
from data import make_optuna_subsample, make_final_subsample
from metrics import find_best_f1_threshold, evaluate_scores, minmax_scale_scores
from results import build_experiment_record, save_record_json, get_memory_mb
from ensemble_utils import (
    tune_if, tune_lof, tune_dbscan, tune_ocsvm,
    score_if, score_lof, score_dbscan, score_ocsvm,
    tune_meta_fusion, apply_meta_fusion,
)

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS = ["CICIDS", "UNSW_NB15"]
SEED = 29
N_TRIALS = 200
META_TRIALS = 60
FUSION_STRATEGIES = ["mean", "max", "weighted", "rank_mean", "stacking_lr"]
DATASET_VERSION = "v1"
PREPROCESSING_VERSION = "v1"
SPLIT_METHOD = "stratified_train_val_test_fixed_seed"
RUN_CONFIGS = [
    dict(run_index=1, n_train_opt=7000,  n_val_opt=3000,  n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run1_small_opt_sample"),
    dict(run_index=2, n_train_opt=21000, n_val_opt=9000,  n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run2_medium_opt_sample"),
    dict(run_index=3, n_train_opt=35000, n_val_opt=15000, n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run3_large_opt_sample"),
]

MODEL_TYPE = "Ensemble_without_LUNAR_v2"
BASE_MODELS = ["IF", "LOF", "DBSCAN", "OCSVM"]
TUNERS = {"IF": tune_if, "LOF": tune_lof, "DBSCAN": tune_dbscan, "OCSVM": tune_ocsvm}
SCORERS = {
    "IF": lambda p, train_x, val_x, test_x: score_if(p, train_x, val_x, test_x, SEED),
    "LOF": lambda p, train_x, val_x, test_x: score_lof(p, train_x, val_x, test_x),
    "DBSCAN": lambda p, train_x, val_x, test_x: score_dbscan(p, train_x, val_x, test_x),
    "OCSVM": lambda p, train_x, val_x, test_x: score_ocsvm(p, train_x, val_x, test_x),
}

records = []
for dataset in DATASETS:
    for run_cfg in RUN_CONFIGS:
        opt_train_x, opt_train_y, opt_val_x, opt_val_y = make_optuna_subsample(dataset, SEED, run_cfg["n_train_opt"], run_cfg["n_val_opt"])
        best_params = {m: TUNERS[m](opt_train_x, opt_val_x, opt_val_y, SEED, N_TRIALS, RESULTS_DIR) for m in BASE_MODELS}
        train_x, train_y, val_x, val_y, test_x, test_y = make_final_subsample(dataset, SEED, run_cfg["n_train_final"], run_cfg["n_val_final"], run_cfg["n_test_final"])
        val_cols, test_cols, runt_train, runt_inf = [], [], 0.0, 0.0
        for model_name in BASE_MODELS:
            v, t, tr, inf = SCORERS[model_name](best_params[model_name], train_x, val_x, test_x)
            val_cols.append(v); test_cols.append(t); runt_train += tr; runt_inf += inf
        val_matrix = np.column_stack(val_cols)
        test_matrix = np.column_stack(test_cols)
        best_meta = tune_meta_fusion(val_matrix, val_y, SEED, META_TRIALS, RESULTS_DIR, FUSION_STRATEGIES)
        fused_val, fused_test = apply_meta_fusion(best_meta, val_matrix, test_matrix, val_y, SEED)
        thr, *_ = find_best_f1_threshold(val_y, fused_val)
        rec = build_experiment_record(dataset, DATASET_VERSION, SPLIT_METHOD, SEED, PREPROCESSING_VERSION, MODEL_TYPE, best_meta["fusion_strategy"], {"base_models": BASE_MODELS, "best_params": best_params, "meta": best_meta}, thr, fused_test, test_y, runt_train, runt_inf, get_memory_mb(), run_cfg["notes"])
        save_record_json(rec, RESULTS_DIR, run_cfg["run_index"], MODEL_TYPE, dataset)
        records.append(rec)

pd.DataFrame(records)
